# LeetCode 942 — DI String Match bằng Python

**Link đề:** https://leetcode.com/problems/di-string-match/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers, String, Greedy (tham lam)

```python
class Solution:
    def diStringMatch(self, s: str) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu **hoán vị** (permutation) của `0..n` và cách chuỗi `I`/`D` mô tả quan hệ giữa hai phần tử kề nhau.
2. Viết **validator** cho bài có nhiều đáp án đúng.
3. Đi từ **vét cạn** (thử mọi hoán vị bằng `itertools.permutations`) → **tham lam hai con trỏ** `lo`/`hi` → **đảo các đoạn D** → **stack**.
4. Hiểu vì sao tham lam đúng (luôn còn "chỗ" để đi lên/đi xuống), trace từng bước.
5. Luyện biến thể: hoán vị nhỏ nhất theo thứ tự từ điển (LeetCode 484), đếm số hoán vị hợp lệ (LeetCode 903 — bản nhỏ).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — dựng hoán vị khớp chuỗi tăng/giảm

Một hoán vị `perm` gồm `n + 1` số nguyên — **tất cả** các số `0, 1, ..., n`, mỗi số đúng một lần — được biểu diễn bởi chuỗi `s` dài `n`:
- `s[i] == 'I'` (Increase) nếu `perm[i] < perm[i + 1]`.
- `s[i] == 'D'` (Decrease) nếu `perm[i] > perm[i + 1]`.

Cho `s`, dựng lại **bất kỳ** `perm` hợp lệ.

| `s` | Một output | Kiểm tra |
|---|---|---|
| `"IDID"` | `[0,4,1,3,2]` | 0<4, 4>1, 1<3, 3>2 ✓ |
| `"III"` | `[0,1,2,3]` | tăng liên tục ✓ |
| `"DDI"` | `[3,2,0,1]` | 3>2, 2>0, 0<1 ✓ |

**Constraints:** `1 <= s.length <= 10^5`; `s[i]` là `'I'` hoặc `'D'`.

Chú ý: output có **`n + 1`** phần tử (nhiều hơn `s` một phần tử) và phải dùng **đúng** các số `0..n`.

## 2. Nền tảng Python cần dùng

### 2.1. Hoán vị và kiểm tra "là hoán vị của 0..n"

Một list là hoán vị của `0..n` ⇔ `sorted(perm) == list(range(n + 1))`.

In [1]:
print(sorted([0, 4, 1, 3, 2]) == list(range(5)))
print(sorted([0, 4, 4, 3, 2]) == list(range(5)), "<- 4 lặp, thiếu 1")

True
False <- 4 lặp, thiếu 1


### 2.2. Validator cho bài nhiều đáp án

`is_valid(s, perm)`:
1. Độ dài `len(s) + 1`.
2. Là hoán vị của `0..n`.
3. Với mọi `i`: nếu `s[i] == 'I'` thì `perm[i] < perm[i+1]`, ngược lại `perm[i] > perm[i+1]`.

In [2]:
def is_valid(s: str, perm: list[int]) -> bool:
    n = len(s)
    if len(perm) != n + 1 or sorted(perm) != list(range(n + 1)):
        return False
    for i, ch in enumerate(s):
        if ch == "I" and not perm[i] < perm[i + 1]:
            return False
        if ch == "D" and not perm[i] > perm[i + 1]:
            return False
    return True

print(is_valid("IDID", [0, 4, 1, 3, 2]), is_valid("IDID", [1, 3, 0, 4, 2]), is_valid("IDID", [0, 1, 2, 3, 4]))

True True False


Chú ý `[1,3,0,4,2]` cũng hợp lệ với `"IDID"` — một bài có **nhiều** đáp án.

### 2.3. `itertools.permutations` — liệt kê mọi hoán vị

`permutations(range(3))` sinh `(0,1,2), (0,2,1), (1,0,2), ...` — tổng cộng `3! = 6` hoán vị. Số hoán vị tăng **cực nhanh**: `10! = 3 628 800`.

In [3]:
from itertools import permutations
import math
print(list(permutations(range(3))))
for k in [3, 5, 8, 10, 12]:
    print(f"{k}! = {math.factorial(k):,}")

[(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]
3! = 6
5! = 120
8! = 40,320
10! = 3,628,800
12! = 479,001,600


`f"{x:,}"` in số có dấu phẩy ngăn cách hàng nghìn.

### 2.4. Đảo một đoạn list bằng gán slice

`a[l:r] = a[l:r][::-1]` đảo ngược đoạn `a[l], ..., a[r-1]` tại chỗ.

In [4]:
a = [0, 1, 2, 3, 4, 5]
a[1:4] = a[1:4][::-1]
print(a)

[0, 3, 2, 1, 4, 5]


## 3. Cách trực giác — vét cạn mọi hoán vị

Thử từng hoán vị của `0..n`, trả về cái đầu tiên hợp lệ. Đúng nhưng `O((n+1)! · n)` — chỉ dùng được với `n` rất nhỏ (≤ 8). Ta vẫn viết nó vì đây là **oracle** (lời giải chắc chắn đúng) để đối chiếu.

In [5]:
def di_bruteforce(s: str) -> list[int]:
    n = len(s)
    for p in permutations(range(n + 1)):
        if is_valid(s, list(p)):
            return list(p)
    return []

for s in ["IDID", "III", "DDI"]:
    print(s, "->", di_bruteforce(s))

IDID -> [0, 2, 1, 4, 3]
III -> [0, 1, 2, 3]
DDI -> [2, 1, 0, 3]


Thú vị: vét cạn trả về hoán vị **nhỏ nhất theo thứ tự từ điển** (vì `permutations` sinh theo thứ tự đó) — chúng khác output mẫu của đề nhưng vẫn đúng.

## 4. Nút thắt và insight — tham lam với hai đầu `lo` và `hi`

Vét cạn quá chậm. Ta cần dựng **trực tiếp** từng phần tử.

**Câu hỏi then chốt:** đặt số nào ở vị trí `i` để **chắc chắn** thỏa `s[i]` bất kể tương lai?
- Nếu `s[i] == 'I'` (phần tử sau phải **lớn hơn**): đặt số **nhỏ nhất còn lại** `lo`. Mọi số còn lại đều lớn hơn `lo` → bước tiếp theo **chắc chắn** đi lên.
- Nếu `s[i] == 'D'` (phần tử sau phải **nhỏ hơn**): đặt số **lớn nhất còn lại** `hi`. Mọi số còn lại đều nhỏ hơn `hi` → chắc chắn đi xuống.

Các số còn lại luôn là một **đoạn liên tiếp** `[lo, hi]` → chỉ cần hai biến. Cuối cùng còn đúng một số (`lo == hi`), đặt nó vào vị trí cuối.

In [6]:
# Thử bằng tay với "IDID": lo=0, hi=4
lo, hi = 0, 4
perm = []
for ch in "IDID":
    if ch == "I":
        perm.append(lo); lo += 1
    else:
        perm.append(hi); hi -= 1
    print(f"{ch}: perm={perm}, còn [{lo}..{hi}]")
perm.append(lo)
print("cuối: perm =", perm)

I: perm=[0], còn [1..4]
D: perm=[0, 4], còn [1..3]
I: perm=[0, 4, 1], còn [2..3]
D: perm=[0, 4, 1, 3], còn [2..2]
cuối: perm = [0, 4, 1, 3, 2]


## 5. Lời giải chính — tham lam hai con trỏ

**Mảnh 1:** `lo, hi = 0, len(s)`, `perm = []`.

**Mảnh 2:** duyệt `ch` trong `s`:
- `'I'` → `perm.append(lo); lo += 1`.
- `'D'` → `perm.append(hi); hi -= 1`.

**Mảnh 3:** sau vòng lặp `lo == hi`, `perm.append(lo)`.

**Mảnh 4:** `return perm`.

In [7]:
def di_greedy(s: str) -> list[int]:
    lo, hi = 0, len(s)
    perm = []
    for ch in s:
        if ch == "I":
            perm.append(lo)
            lo += 1
        else:
            perm.append(hi)
            hi -= 1
    perm.append(lo)          # lúc này lo == hi
    return perm

for s in ["IDID", "III", "DDI"]:
    out = di_greedy(s)
    print(s, "->", out, "| hợp lệ:", is_valid(s, out))

IDID -> [0, 4, 1, 3, 2] | hợp lệ: True
III -> [0, 1, 2, 3] | hợp lệ: True
DDI -> [3, 2, 0, 1] | hợp lệ: True


## 6. TRACE từng bước

Ví dụ `s = "DDIID"` (n = 5, dùng số `0..5`):

| i | s[i] | lo | hi | Đặt | perm sau | Còn lại |
|:---:|:---:|:---:|:---:|:---:|---|---|
| 0 | D | 0 | 5 | hi=5 | [5] | 0..4 |
| 1 | D | 0 | 4 | hi=4 | [5,4] | 0..3 |
| 2 | I | 0 | 3 | lo=0 | [5,4,0] | 1..3 |
| 3 | I | 1 | 3 | lo=1 | [5,4,0,1] | 2..3 |
| 4 | D | 2 | 3 | hi=3 | [5,4,0,1,3] | 2..2 |
| cuối | | 2 | 2 | 2 | [5,4,0,1,3,2] | — |

Kiểm tra: 5>4 (D), 4>0 (D), 0<1 (I), 1<3 (I), 3>2 (D) ✓

In [8]:
def trace_di(s):
    lo, hi = 0, len(s)
    perm = []
    print(f"s = {s!r}, các số 0..{hi}")
    for i, ch in enumerate(s):
        if ch == "I":
            pick = lo; lo += 1
            why = "I -> lấy nhỏ nhất (mọi số sau đều lớn hơn)"
        else:
            pick = hi; hi -= 1
            why = "D -> lấy lớn nhất (mọi số sau đều nhỏ hơn)"
        perm.append(pick)
        print(f"  i={i} {why}: đặt {pick} | perm={perm} | còn [{lo}..{hi}]")
    perm.append(lo)
    print(f"  cuối: còn đúng số {lo} -> perm={perm} | hợp lệ: {is_valid(s, perm)}")
    return perm

trace_di("DDIID")

s = 'DDIID', các số 0..5
  i=0 D -> lấy lớn nhất (mọi số sau đều nhỏ hơn): đặt 5 | perm=[5] | còn [0..4]
  i=1 D -> lấy lớn nhất (mọi số sau đều nhỏ hơn): đặt 4 | perm=[5, 4] | còn [0..3]
  i=2 I -> lấy nhỏ nhất (mọi số sau đều lớn hơn): đặt 0 | perm=[5, 4, 0] | còn [1..3]
  i=3 I -> lấy nhỏ nhất (mọi số sau đều lớn hơn): đặt 1 | perm=[5, 4, 0, 1] | còn [2..3]
  i=4 D -> lấy lớn nhất (mọi số sau đều nhỏ hơn): đặt 3 | perm=[5, 4, 0, 1, 3] | còn [2..2]
  cuối: còn đúng số 2 -> perm=[5, 4, 0, 1, 3, 2] | hợp lệ: True


[5, 4, 0, 1, 3, 2]

## 7. Các lời giải khác và bảng so sánh

### 7.1. Đảo các đoạn `D` trên dãy `0..n`

Bắt đầu từ `[0, 1, ..., n]` (thỏa mọi `I`). Với mỗi **đoạn liên tiếp các `D`** từ vị trí `l` đến `r-1` trong `s`, đảo ngược đoạn `perm[l..r]` (gồm `r - l + 1` phần tử). Đoạn tăng biến thành giảm, và hai đầu đoạn vẫn khớp với `I` xung quanh.

Cách này cho **hoán vị nhỏ nhất theo thứ tự từ điển** — chính là đáp án của LeetCode 484.

In [9]:
def di_reverse_runs(s: str) -> list[int]:
    n = len(s)
    perm = list(range(n + 1))
    i = 0
    while i < n:
        if s[i] == "D":
            j = i
            while j < n and s[j] == "D":
                j += 1
            perm[i:j + 1] = perm[i:j + 1][::-1]   # đảo đoạn gồm j - i + 1 phần tử
            i = j
        else:
            i += 1
    return perm

for s in ["IDID", "III", "DDI", "DDIID"]:
    out = di_reverse_runs(s)
    print(s, "->", out, "| hợp lệ:", is_valid(s, out), "| = vét cạn (nhỏ nhất)?", out == di_bruteforce(s))

IDID -> [0, 2, 1, 4, 3] | hợp lệ: True | = vét cạn (nhỏ nhất)? True
III -> [0, 1, 2, 3] | hợp lệ: True | = vét cạn (nhỏ nhất)? True
DDI -> [2, 1, 0, 3] | hợp lệ: True | = vét cạn (nhỏ nhất)? True
DDIID -> [2, 1, 0, 3, 5, 4] | hợp lệ: True | = vét cạn (nhỏ nhất)? True


### 7.2. Dùng stack

Duyệt `i = 0..n`, đẩy `i` vào stack. Khi gặp `'I'` hoặc hết chuỗi, **đổ hết stack** ra kết quả (stack trả ngược thứ tự → tạo đoạn giảm cho các `D` đứng trước). Kết quả giống cách đảo đoạn.

In [10]:
def di_stack(s: str) -> list[int]:
    res, stack = [], []
    for i in range(len(s) + 1):
        stack.append(i)
        if i == len(s) or s[i] == "I":
            while stack:
                res.append(stack.pop())
    return res

for s in ["IDID", "III", "DDI", "DDIID"]:
    out = di_stack(s)
    print(s, "->", out, "| hợp lệ:", is_valid(s, out))

IDID -> [0, 2, 1, 4, 3] | hợp lệ: True
III -> [0, 1, 2, 3] | hợp lệ: True
DDI -> [2, 1, 0, 3] | hợp lệ: True
DDIID -> [2, 1, 0, 3, 5, 4] | hợp lệ: True


### 7.3. Tham lam ngược: `D` lấy nhỏ ... không được!

Thử "ngược" logic: `'I'` lấy **lớn nhất**, `'D'` lấy **nhỏ nhất**. Sai — vì đặt số lớn nhất rồi cần đi **lên** thì không còn số nào lớn hơn. Đây là cách tự kiểm tra hiểu insight.

In [11]:
def di_greedy_backwards(s):
    lo, hi = 0, len(s)
    perm = []
    for ch in s:
        if ch == "I":
            perm.append(hi); hi -= 1
        else:
            perm.append(lo); lo += 1
    perm.append(lo)
    return perm

out = di_greedy_backwards("IDID")
print(out, "| hợp lệ:", is_valid("IDID", out))

[4, 0, 3, 1, 2] | hợp lệ: False


### 7.4. Bảng so sánh

`n = len(s)`.

| Cách | Time | Space phụ | Đáp án | Ghi chú |
|---|---|---|---|---|
| Vét cạn hoán vị | `O((n+1)!·n)` | `O(n)` | nhỏ nhất từ điển | chỉ làm oracle |
| **Tham lam lo/hi** | `O(n)` | `O(1)` (ngoài output) | một đáp án hợp lệ | nên nộp |
| Đảo đoạn D | `O(n)` | `O(1)` (ngoài output) | nhỏ nhất từ điển | dùng lại cho LC 484 |
| Stack | `O(n)` | `O(n)` | nhỏ nhất từ điển | cùng ý đảo đoạn |

## 8. Code nộp LeetCode

In [12]:
class Solution:
    def diStringMatch(self, s: str) -> list[int]:
        lo, hi = 0, len(s)
        perm = []
        for ch in s:
            if ch == "I":
                perm.append(lo)
                lo += 1
            else:
                perm.append(hi)
                hi -= 1
        perm.append(lo)
        return perm

sol = Solution()
for s in ["IDID", "III", "DDI"]:
    out = sol.diStringMatch(s)
    assert is_valid(s, out), (s, out)
    print(s, "->", out)
print("Solution PASS 3 ví dụ (kiểm tra bằng validator)")

IDID -> [0, 4, 1, 3, 2]
III -> [0, 1, 2, 3]
DDI -> [3, 2, 0, 1]
Solution PASS 3 ví dụ (kiểm tra bằng validator)


Đáp án tham lam trùng đúng output mẫu của đề (`[0,4,1,3,2]`, `[0,1,2,3]`, `[3,2,0,1]`) — gợi ý đây là lời giải "chính thức".

## 9. Vì sao thuật toán đúng?

**Invariant:** trước khi xử lý `s[i]`, tập số **chưa dùng** đúng là đoạn liên tiếp `{lo, lo+1, ..., hi}` và `hi - lo = n - i`.

- Ban đầu `i = 0`: chưa dùng `{0..n}`, `hi - lo = n` ✓.
- Lấy `lo` (hoặc `hi`) ra khỏi một đầu → phần còn lại vẫn liên tiếp, khoảng cách giảm 1 ✓.

**Mỗi ký tự được thỏa:** giả sử tại bước `i` ta đặt `perm[i]`.
- Nếu `s[i] = 'I'`: `perm[i] = lo` là **nhỏ nhất** trong tập còn lại tại thời điểm đó. `perm[i+1]` sẽ được chọn từ tập còn lại sau đó, toàn số `> lo` → `perm[i] < perm[i+1]` ✓.
- Nếu `s[i] = 'D'`: `perm[i] = hi` là **lớn nhất**, mọi số sau đều `< hi` ✓.

**Là hoán vị:** mỗi số được lấy đúng một lần từ tập chưa dùng; sau `n` bước còn đúng 1 số (`lo == hi`), đặt vào cuối → dùng đủ `n + 1` số. ∎

Điểm hay: quyết định ở bước `i` **không phụ thuộc tương lai** — đó là đặc trưng của thuật toán **tham lam** đúng.

In [13]:
# Kiểm chứng invariant trong lúc chạy
def checked_greedy(s):
    lo, hi = 0, len(s)
    used = set()
    perm = []
    for i, ch in enumerate(s):
        remaining = set(range(len(s) + 1)) - used
        assert remaining == set(range(lo, hi + 1)) and hi - lo == len(s) - i
        x = lo if ch == "I" else hi
        if ch == "I": lo += 1
        else: hi -= 1
        perm.append(x); used.add(x)
    perm.append(lo)
    return perm

print(checked_greedy("DDIIDIDDDI"), "- invariant luôn đúng")

[10, 9, 0, 1, 8, 2, 7, 6, 5, 3, 4] - invariant luôn đúng


`set(a) - set(b)` là **hiệu hai tập hợp**: các phần tử thuộc `a` nhưng không thuộc `b`.

## 10. Phân tích Time / Space

- **Time `O(n)`:** một lượt qua `s`, mỗi bước `O(1)` (`append` khấu hao `O(1)`).
- **Space `O(1)` phụ** (chỉ `lo`, `hi`); mảng kết quả `O(n)` là bắt buộc vì phải trả về.

In [14]:
import time, random
rng = random.Random(942)
big = "".join(rng.choice("ID") for _ in range(10**5))
for f in [di_greedy, di_reverse_runs, di_stack]:
    t0 = time.perf_counter()
    out = f(big)
    dt = time.perf_counter() - t0
    print(f"{f.__name__:16} {dt:.4f}s | hợp lệ: {is_valid(big, out)}")

di_greedy        0.0060s | hợp lệ: True
di_reverse_runs  0.0144s | hợp lệ: True


di_stack         0.0121s | hợp lệ: True


## 11. Bộ test — mẫu đề, biên, đối chiếu

Nhóm test (tất cả kiểm tra bằng validator):
1. Ba ví dụ đề.
2. Một ký tự: `"I"`, `"D"`.
3. Toàn `I`, toàn `D`.
4. Xen kẽ, đoạn `D` dài.
5. Ngẫu nhiên có seed; với `n <= 7` đối chiếu thêm "nhỏ nhất từ điển" giữa vét cạn, đảo đoạn, stack.

In [15]:
tests = ["IDID", "III", "DDI", "I", "D", "IIIII", "DDDDD", "IDIDIDID", "DIDIDI", "IIDDDDI", "DDDDDDI"]
solvers = [di_greedy, di_reverse_runs, di_stack, Solution().diStringMatch]
for s in tests:
    for f in solvers:
        out = f(s)
        assert is_valid(s, out), (f.__name__, s, out)
print(f"PASS {len(tests)} test x {len(solvers)} cách")
assert di_greedy("IDID") == [0, 4, 1, 3, 2] and di_greedy("III") == [0, 1, 2, 3] and di_greedy("DDI") == [3, 2, 0, 1]
print("Tham lam trùng output mẫu của đề")

PASS 11 test x 4 cách
Tham lam trùng output mẫu của đề


In [16]:
for _ in range(300):
    s = "".join(rng.choice("ID") for _ in range(rng.randint(1, 7)))
    for f in solvers:
        assert is_valid(s, f(s)), (f.__name__, s)
    smallest = di_bruteforce(s)
    assert di_reverse_runs(s) == smallest and di_stack(s) == smallest, s
for _ in range(200):
    s = "".join(rng.choice("ID") for _ in range(rng.randint(1, 2000)))
    for f in solvers:
        assert is_valid(s, f(s)), (f.__name__, len(s))
print("PASS 500 chuỗi ngẫu nhiên (kể cả đối chiếu thứ tự từ điển với vét cạn)")

PASS 500 chuỗi ngẫu nhiên (kể cả đối chiếu thứ tự từ điển với vét cạn)


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên phần tử cuối (`perm` chỉ có `n` phần tử)

In [17]:
def wrong_missing_last(s):
    lo, hi = 0, len(s)
    perm = []
    for ch in s:
        if ch == "I":
            perm.append(lo); lo += 1
        else:
            perm.append(hi); hi -= 1
    return perm                    # quên append(lo)

out = wrong_missing_last("IDID")
print(out, "len =", len(out), "| hợp lệ:", is_valid("IDID", out))

[0, 4, 1, 3] len = 4 | hợp lệ: False


**Sửa:** sau vòng lặp `perm.append(lo)`.

### Lỗi 2: Khởi tạo `hi = len(s) - 1`

In [18]:
def wrong_hi(s):
    lo, hi = 0, len(s) - 1          # SAI: phải dùng các số 0..n, n = len(s)
    perm = []
    for ch in s:
        if ch == "I":
            perm.append(lo); lo += 1
        else:
            perm.append(hi); hi -= 1
    perm.append(lo)
    return perm

out = wrong_hi("IDID")
print(out, "| hợp lệ:", is_valid("IDID", out))

[0, 3, 1, 2, 2] | hợp lệ: False


**Vì sao sai:** hoán vị có `n + 1` số `0..n`, số lớn nhất là `n = len(s)`. Thiếu số `n` và lặp một số khác.

### Lỗi 3: Đảo logic tham lam (`I` lấy lớn, `D` lấy nhỏ)

Đã thấy ở 7.3 — vi phạm ngay ký tự đầu tiên.

### Lỗi 4: Test bằng so sánh với một đáp án cố định

Cách đảo đoạn trả `[0,2,1,4,3]` cho `"IDID"`, khác `[0,4,1,3,2]` nhưng **đúng**. Luôn dùng `is_valid`.

In [19]:
out = di_reverse_runs("IDID")
print(out, "== [0,4,1,3,2]?", out == [0, 4, 1, 3, 2], "| hợp lệ:", is_valid("IDID", out))

[0, 2, 1, 4, 3] == [0,4,1,3,2]? False | hợp lệ: True


### Lỗi 5: Vét cạn với `n` lớn

`permutations` với `n = 12` đã có hơn 6 tỷ hoán vị (13!). Chỉ dùng vét cạn làm oracle cho `n` nhỏ.

## 13. Mẹo nhận dạng pattern

- Dựng một dãy thỏa các ràng buộc **cục bộ giữa hai phần tử kề nhau**, dùng các số `0..n` → nghĩ tới **tham lam hai đầu**: muốn bước sau đi lên thì đặt nhỏ nhất, đi xuống thì đặt lớn nhất.
- "Các số còn lại luôn là một đoạn liên tiếp" → chỉ cần **hai con trỏ** `lo`, `hi`.
- Cần đáp án **nhỏ nhất theo thứ tự từ điển** → bắt đầu từ dãy tăng `0..n` rồi **đảo các đoạn D** (hoặc stack).
- Bài nhiều đáp án → **validator**.

Bài liên quan: 484 Find Permutation, 2375 Construct Smallest Number From DI String, 903 Valid Permutations for DI Sequence (đếm — DP), 667 Beautiful Arrangement II.

## 14. Biến thể

### 14.1. LeetCode 2375 — số nhỏ nhất dùng chữ số 1..9 không lặp
Chính là "đảo đoạn D" trên dãy `1..n+1`, rồi ghép thành chuỗi.

In [20]:
def smallest_number(pattern: str) -> str:
    return "".join(str(x + 1) for x in di_reverse_runs(pattern))

print(smallest_number("IIIDIDDD"), smallest_number("DDD"))   # 123549876, 4321

123549876 4321


### 14.2. Hoán vị **lớn nhất** theo thứ tự từ điển

Đối xứng: bắt đầu từ dãy giảm `n..0` và đảo các đoạn `I`.

In [21]:
def di_largest(s: str) -> list[int]:
    n = len(s)
    perm = list(range(n, -1, -1))
    i = 0
    while i < n:
        if s[i] == "I":
            j = i
            while j < n and s[j] == "I":
                j += 1
            perm[i:j + 1] = perm[i:j + 1][::-1]
            i = j
        else:
            i += 1
    return perm

for s in ["IDID", "DDI"]:
    out = di_largest(s)
    biggest = max(list(p) for p in permutations(range(len(s) + 1)) if is_valid(s, list(p)))
    print(s, "->", out, "| hợp lệ:", is_valid(s, out), "| lớn nhất?", out == biggest)

IDID -> [3, 4, 1, 2, 0] | hợp lệ: True | lớn nhất? True
DDI -> [3, 2, 0, 1] | hợp lệ: True | lớn nhất? True


### 14.3. Đếm số hoán vị hợp lệ (LeetCode 903, bản nhỏ bằng vét cạn)

In [22]:
def count_valid_bruteforce(s):
    return sum(1 for p in permutations(range(len(s) + 1)) if is_valid(s, list(p)))

for s in ["I", "D", "ID", "DID", "IDID"]:
    print(s, "->", count_valid_bruteforce(s))

I -> 1
D -> 1
ID -> 2
DID -> 5
IDID -> 16


## 15. Bài tập tự luyện (TODO — không tự chạy)

### Bài 1 — Tính tay
Chạy tham lam `lo/hi` bằng tay cho `s = "IIDDI"`. Kết quả là gì?

### Bài 2 — Viết validator

In [23]:
def exercise_is_valid(s: str, perm: list[int]) -> bool:
    # TODO: độ dài, là hoán vị 0..n, khớp từng ký tự
    pass

# assert exercise_is_valid("IDID", [0, 4, 1, 3, 2]) == True
# assert exercise_is_valid("IDID", [0, 1, 2, 3, 4]) == False

### Bài 3 — Tự viết lời giải tham lam 942

In [24]:
def exercise_di_string_match(s: str) -> list[int]:
    # TODO: lo, hi; I -> lo, D -> hi; cuối thêm lo
    pass

# for s in ["IDID", "III", "DDI"]: assert is_valid(s, exercise_di_string_match(s))

### Bài 4 — Hoán vị nhỏ nhất từ điển (LeetCode 484 dùng số 1..n+1)

In [25]:
def exercise_find_permutation(s: str) -> list[int]:
    # TODO: dãy 1..n+1, đảo các đoạn D
    pass

# assert exercise_find_permutation("I") == [1, 2]
# assert exercise_find_permutation("DI") == [2, 1, 3]

### Bài 5 — Từ hoán vị suy ra chuỗi DI (bài toán ngược)

In [26]:
def exercise_perm_to_di(perm: list[int]) -> str:
    # TODO
    pass

# assert exercise_perm_to_di([0, 4, 1, 3, 2]) == "IDID"
# assert exercise_perm_to_di([3, 2, 0, 1]) == "DDI"

### Bài 6 — Mini challenge: đếm số hoán vị hợp lệ bằng DP (LeetCode 903)

Gợi ý: `dp[i][j]` = số cách xếp `i + 1` số đầu tiên (dùng **thứ hạng tương đối** `0..i`) sao cho phần tử cuối có thứ hạng `j`. Với `'I'`: `dp[i][j] = sum(dp[i-1][k] for k < j)`; với `'D'`: `dp[i][j] = sum(dp[i-1][k] for k >= j)`.

In [27]:
def exercise_count_di(s: str) -> int:
    # TODO
    pass

# assert exercise_count_di("DID") == 5
# assert exercise_count_di("IDID") == count_valid_bruteforce("IDID")

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | lo=0, hi=5 | I,I lấy 0,1; D,D lấy 5,4 | I lấy 2; cuối 3 |
| 2 | 3 điều kiện | `sorted(perm) == list(range(n+1))` | so sánh theo `s[i]` |
| 3 | I → nhỏ nhất | `perm.append(lo); lo += 1` | nhớ `perm.append(lo)` cuối |
| 4 | dãy tăng thỏa mọi I | tìm đoạn D `[i, j)` | đảo `perm[i:j+1]` |
| 5 | so sánh cặp kề | `zip(perm, perm[1:])` | `"I" if a < b else "D"` |
| 6 | thứ hạng tương đối | tổng tiền tố | `D`: `dp[i][j] = sum(dp[i-1][j:i])` |

## 17. Đáp án tham khảo

Bài 1: `"IIDDI"` → `[0, 1, 5, 4, 2, 3]`.

In [28]:
print(di_greedy("IIDDI"), is_valid("IIDDI", di_greedy("IIDDI")))

[0, 1, 5, 4, 2, 3] True


In [29]:
def answer_is_valid(s, perm):
    n = len(s)
    return (len(perm) == n + 1 and sorted(perm) == list(range(n + 1))
            and all((a < b_) == (ch == "I") for ch, a, b_ in zip(s, perm, perm[1:])))

def answer_di_string_match(s):
    lo, hi, perm = 0, len(s), []
    for ch in s:
        if ch == "I":
            perm.append(lo); lo += 1
        else:
            perm.append(hi); hi -= 1
    return perm + [lo]

def answer_find_permutation(s):
    n = len(s)
    perm = list(range(1, n + 2))
    i = 0
    while i < n:
        if s[i] == "D":
            j = i
            while j < n and s[j] == "D":
                j += 1
            perm[i:j + 1] = perm[i:j + 1][::-1]
            i = j
        else:
            i += 1
    return perm

def answer_perm_to_di(perm):
    return "".join("I" if a < b_ else "D" for a, b_ in zip(perm, perm[1:]))

def answer_count_di(s):
    n = len(s)
    dp = [1]                                   # 1 phần tử: thứ hạng 0
    for i in range(1, n + 1):
        new = [0] * (i + 1)
        for j in range(i + 1):
            if s[i - 1] == "I":
                new[j] = sum(dp[:j])           # phần tử trước có hạng < j
            else:
                new[j] = sum(dp[j:])           # phần tử trước có hạng >= j
        dp = new
    return sum(dp)

assert answer_is_valid("IDID", [0, 4, 1, 3, 2]) and not answer_is_valid("IDID", [0, 1, 2, 3, 4])
for s in ["IDID", "III", "DDI"]:
    assert is_valid(s, answer_di_string_match(s))
assert answer_find_permutation("I") == [1, 2] and answer_find_permutation("DI") == [2, 1, 3]
assert answer_perm_to_di([0, 4, 1, 3, 2]) == "IDID" and answer_perm_to_di([3, 2, 0, 1]) == "DDI"
assert answer_count_di("DID") == 5
for s in ["I", "D", "ID", "DID", "IDID", "DDIID", "IIDDI"]:
    assert answer_count_di(s) == count_valid_bruteforce(s), s
for s in tests:
    assert is_valid(s, answer_di_string_match(s))
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


Giải thích nhanh Bài 6: khi thêm phần tử thứ `i + 1` có thứ hạng `j` trong `i + 1` phần tử, các phần tử cũ có thứ hạng `>= j` được "đẩy lên" 1. Vì vậy với `'D'` (phần tử trước lớn hơn), thứ hạng cũ `k` phải thỏa `k >= j`; với `'I'`, `k < j`.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def diStringMatch(self, s: str) -> list[int]:
        lo, hi = 0, len(s)
        perm = []
        for ch in s:
            if ch == "I":
                perm.append(lo); lo += 1     # sau đó chắc chắn lớn hơn
            else:
                perm.append(hi); hi -= 1     # sau đó chắc chắn nhỏ hơn
        perm.append(lo)                      # lo == hi
        return perm
```

**Câu thần chú:** *"I lấy nhỏ nhất, D lấy lớn nhất, cuối cùng thêm số còn lại."*

**Checklist trước khi submit:**
- [ ] `hi = len(s)` (không phải `len(s) - 1`).
- [ ] Output có `len(s) + 1` phần tử.
- [ ] Không đảo logic I/D.
- [ ] Test bằng validator, không so với một đáp án cố định.